In [ ]:
import pandas as pd
import numpy as np
import sys

def exit_with_error(message):
    print(f"Error: {message}")
    sys.exit(1)

metadata_fname = "metadata/metadata_wide.tsv"
quants_fname = "data/proteins_normalized_wide.tsv"

metadata = pd.read_csv(metadata_fname, sep='\t', na_values=['NA'])
data = pd.read_csv(quants_fname, sep='\t')
data = data.fillna(0)
protein_names = data['protein'].values

print("Metadata shape:", metadata.shape)
print("Data shape:", data.shape)

In [ ]:
import matplotlib.pyplot as plt

valid_replicates = metadata[metadata['Age_months'].notna()]['replicate'].tolist()

X = []
month = []
sex = []

for idx in valid_replicates:
  for sample in data.columns[:]:
    if idx in sample:
      s_month = metadata[metadata['replicate'] == idx]['Age_months'].values[0]
      s_sex = metadata[metadata['replicate'] == idx]['Sex'].values[0]

      month.append(s_month)
      sex.append(s_sex)
      X.append(data[sample].values)

X = np.array(X)
X = np.power(2, X) # "unlog" X

month = np.array(month)
month = month.astype(float)

sex = np.array(sex)
sex = np.where(sex == "Male", 0, 1) # encode the sex array
sex = sex.astype(int)

print("Data shape:", X.shape)
print("Months:", month)
print("Sex:", sex)

total_signal = np.sum(X, axis=1)
plt.hist(np.log10(total_signal), bins = 100)
plt.title('Distribution of total signal for each sample')
plt.ylabel('Number of samples')
plt.xlabel('Sum of peptide quants (log10)')
plt.show()

In [ ]:
# Create the box plot
plt.figure(figsize=(15, 8))
plt.boxplot(X.T)  # Transpose the data

plt.title('Raw Protein Quantites by Sample')
plt.xlabel('Samples')
plt.ylabel('Peptide Quantity (Log)')
plt.xticks(range(1, X.shape[0] + 1), [f'Sample {i+1}' for i in range(X.shape[0])], rotation=45, ha='right')

# Use log scale for y-axis as peptide quantities often span several orders of magnitude
plt.yscale('log')

plt.tight_layout()  # Adjust layout to prevent clipping of labels
plt.show()

In [ ]:
from __future__ import annotations
from typing import Dict, Literal, Optional, Union
import numpy as np
from scipy.stats import rankdata, norm, t as student_t
from statsmodels.stats.multitest import multipletests


def spearman_fisherz_sex_diff(
    X: np.ndarray,
    protein_names: np.ndarray,
    sex: np.ndarray,
    timepoint: np.ndarray,
    *,
    male_label: Union[str, int] = "M",
    female_label: Union[str, int] = "F",
    min_n_per_sex: int = 6,
    nan_policy: Literal["pairwise", "complete"] = "pairwise",
    return_type: Literal["dict", "dataframe", "structured"] = "dict",
) -> Union[Dict[str, np.ndarray], "np.ndarray", "pd.DataFrame"]:
    """
    Compute (per protein):
      - global Spearman rho (all samples vs timepoint) + approx p-value
      - sex-specific Spearman rhos (male, female) + approx p-values
      - Fisher z-test for difference between sex-specific rhos (independent groups)
      - p_diff and BH-FDR across proteins for p_diff

    This is faster than calling scipy.stats.spearmanr in a loop because it computes
    Spearman as Pearson correlation of ranks and uses a lightweight correlation/p-value
    calculation. Complexity is ~O(P * N log N) dominated by rank computations.

    Parameters
    ----------
    X : (n_samples, n_proteins) array
    protein_names : (n_proteins,) array of labels
    sex : (n_samples,) array of labels. Supports string labels (e.g. "M"/"F") or
        integer-encoded labels (e.g. 0/1). male_label and female_label must match
        the dtype used (e.g. male_label=1, female_label=0 for integer encoding).
    timepoint : (n_samples,) numeric array (age/time)
    male_label / female_label : labels identifying sexes in `sex`. Can be str or int.
        For integer-encoded sex, pass e.g. male_label=1, female_label=0.
    min_n_per_sex : minimum valid N per sex required to compute z-diff.
        A hard floor of 4 is also enforced because Fisher z requires n-3 > 0.
    nan_policy :
        - "pairwise": per protein, drop rows where X[:,j] or timepoint is non-finite
        - "complete": drop rows where any protein OR timepoint is non-finite (strict)
    return_type :
        - "dict": returns dict of numpy arrays
        - "structured": returns structured numpy array
        - "dataframe": returns pandas DataFrame (requires pandas)

    Notes on p-values
    -----------------
    The Spearman p-values here use a common large-sample approximation:
      treat Spearman rho like Pearson r on ranks and use a t-test with df = n-2.
    This is typically fine for screening thousands of proteins. If you need exact
    p-values for small n, you can swap in scipy.stats.spearmanr per protein.

    Notes on n_all vs n_male + n_female
    ------------------------------------
    In pairwise mode, n_all reflects finite rows across all samples for protein j.
    If any samples have a sex label other than male_label or female_label (e.g.
    "unknown"), n_male + n_female may be less than n_all. This is expected.
    Protein names longer than 256 characters will be silently truncated in the
    structured array return type.

    Returns
    -------
    Per-protein statistics including:
      protein, n_all, n_male, n_female,
      rho_all, p_all,
      rho_male, p_male,
      rho_female, p_female,
      z_diff, p_diff, fdr_diff
    """
    X = np.asarray(X)
    protein_names = np.asarray(protein_names)
    timepoint = np.asarray(timepoint, dtype=float)

    # Normalise sex array: if integer labels are provided, cast sex to the same
    # integer dtype so that equality comparisons work correctly regardless of
    # whether the caller passes a bool, int8, int64, etc. array.
    sex = np.asarray(sex)
    if isinstance(male_label, (int, np.integer)) or isinstance(female_label, (int, np.integer)):
        # Coerce both the array and the labels to a consistent integer type so
        # that, e.g., sex dtype int8 doesn't silently fail to match a Python int.
        sex = sex.astype(int)
        male_label = int(male_label)
        female_label = int(female_label)

    if X.ndim != 2:
        raise ValueError(f"X must be 2D (n_samples, n_proteins); got {X.shape}")
    n_samples, n_proteins = X.shape

    if protein_names.ndim != 1 or protein_names.shape[0] != n_proteins:
        raise ValueError(
            f"protein_names must be 1D length {n_proteins}; got {protein_names.shape}"
        )
    if sex.ndim != 1 or sex.shape[0] != n_samples:
        raise ValueError(f"sex must be 1D length {n_samples}; got {sex.shape}")
    if timepoint.ndim != 1 or timepoint.shape[0] != n_samples:
        raise ValueError(f"timepoint must be 1D length {n_samples}; got {timepoint.shape}")

    male_mask = (sex == male_label)
    female_mask = (sex == female_label)

    if not np.any(male_mask):
        raise ValueError(f"No samples match male_label={male_label!r}")
    if not np.any(female_mask):
        raise ValueError(f"No samples match female_label={female_label!r}")
    if nan_policy not in ("pairwise", "complete"):
        raise ValueError("nan_policy must be 'pairwise' or 'complete'")

    # Optionally drop rows up-front (strict)
    if nan_policy == "complete":
        row_ok = np.isfinite(timepoint) & np.all(np.isfinite(X), axis=1)
        Xw = X[row_ok]
        tw = timepoint[row_ok]
        sexw = sex[row_ok]
        male_mask_w = (sexw == male_label)
        female_mask_w = (sexw == female_label)
    else:
        Xw = X
        tw = timepoint
        sexw = sex
        male_mask_w = male_mask
        female_mask_w = female_mask

    # ---------- helpers ----------

    def _corr_and_p_from_vectors(x: np.ndarray, y: np.ndarray) -> tuple[float, float, int]:
        """
        Compute Spearman rho by:
          - mask finite
          - rank x and y
          - Pearson correlation on ranks
          - approximate p-value using t-test with df=n-2
        Returns (rho, p, n_used)
        """
        m = np.isfinite(x) & np.isfinite(y)
        n = int(m.sum())
        if n < 3:
            return (np.nan, np.nan, n)
        xv = x[m]
        yv = y[m]
        # If constant, correlation undefined
        if np.all(xv == xv[0]) or np.all(yv == yv[0]):
            return (np.nan, np.nan, n)
        rx = rankdata(xv, method="average")
        ry = rankdata(yv, method="average")
        # Pearson correlation of ranks
        rx_mean = rx.mean()
        ry_mean = ry.mean()
        rx0 = rx - rx_mean
        ry0 = ry - ry_mean
        denom = np.sqrt(np.sum(rx0 * rx0) * np.sum(ry0 * ry0))
        if denom == 0:
            return (np.nan, np.nan, n)
        r = float(np.sum(rx0 * ry0) / denom)
        # Approx p-value via t statistic (common approximation)
        # t = r * sqrt((n-2)/(1-r^2)), df=n-2
        # Protect against tiny numerical issues at |r| ~ 1
        r_clamped = np.clip(r, -0.999999999, 0.999999999)
        t_stat = r_clamped * np.sqrt((n - 2) / max(1e-30, (1.0 - r_clamped * r_clamped)))
        p = float(2.0 * student_t.sf(np.abs(t_stat), df=n - 2))
        return (r, p, n)

    def _fisher_z(r: float) -> float:
        if not np.isfinite(r):
            return np.nan
        r = np.clip(r, -0.999999999, 0.999999999)
        return 0.5 * np.log((1.0 + r) / (1.0 - r))

    # ---------- allocate outputs ----------
    protein_str = protein_names.astype(str)
    n_all = np.zeros(n_proteins, dtype=int)
    n_male = np.zeros(n_proteins, dtype=int)
    n_female = np.zeros(n_proteins, dtype=int)
    rho_all = np.full(n_proteins, np.nan, dtype=float)
    p_all = np.full(n_proteins, np.nan, dtype=float)
    rho_m = np.full(n_proteins, np.nan, dtype=float)
    p_m = np.full(n_proteins, np.nan, dtype=float)
    rho_f = np.full(n_proteins, np.nan, dtype=float)
    p_f = np.full(n_proteins, np.nan, dtype=float)
    z_diff = np.full(n_proteins, np.nan, dtype=float)
    p_diff = np.full(n_proteins, np.nan, dtype=float)

    # ---------- main loop (fast rank + corr; avoids scipy spearmanr overhead) ----------
    # Pre-slice group arrays once (saves repeated boolean indexing per protein)
    Xm = Xw[male_mask_w, :]
    tm = tw[male_mask_w]
    Xf = Xw[female_mask_w, :]
    tf = tw[female_mask_w]

    # Fisher z-diff requires n >= 4 in each group (so that n-3 > 0). We also
    # respect the caller-supplied min_n_per_sex, taking whichever is stricter.
    _min_n = max(min_n_per_sex, 4)

    for j in range(n_proteins):
        xj = Xw[:, j]
        r, p, n = _corr_and_p_from_vectors(xj, tw)
        rho_all[j], p_all[j], n_all[j] = r, p, n

        r_m, p_mj, nm = _corr_and_p_from_vectors(Xm[:, j], tm)
        rho_m[j], p_m[j], n_male[j] = r_m, p_mj, nm

        r_f, p_fj, nf = _corr_and_p_from_vectors(Xf[:, j], tf)
        rho_f[j], p_f[j], n_female[j] = r_f, p_fj, nf

        # Fisher z-difference test between independent correlations
        # Z = (z_m - z_f) / sqrt(1/(n_m-3) + 1/(n_f-3))
        if (
            np.isfinite(r_m) and np.isfinite(r_f)
            and nm >= _min_n
            and nf >= _min_n
        ):
            zm = _fisher_z(r_m)
            zf = _fisher_z(r_f)
            se = np.sqrt(1.0 / (nm - 3.0) + 1.0 / (nf - 3.0))
            if np.isfinite(zm) and np.isfinite(zf) and se > 0:
                z = (zm - zf) / se
                # Use norm.sf for better numerical precision in the extreme tails
                # compared to 1 - norm.cdf.
                pz = float(2.0 * norm.sf(np.abs(z)))
                z_diff[j] = z
                p_diff[j] = pz

    # ---------- BH-FDR using a standard library ----------
    fdr_diff = np.full(n_proteins, np.nan, dtype=float)
    ok = np.isfinite(p_diff)
    if np.any(ok):
        # multipletests returns: reject, pvals_corrected, alphacSidak, alphacBonf
        _, p_corr, _, _ = multipletests(p_diff[ok], method="fdr_bh")
        fdr_diff[ok] = p_corr

    # ---------- package results ----------
    if return_type == "dict":
        return {
            "protein": protein_str,
            "n_all": n_all,
            "n_male": n_male,
            "n_female": n_female,
            "rho_all": rho_all,
            "p_all": p_all,
            "rho_male": rho_m,
            "p_male": p_m,
            "rho_female": rho_f,
            "p_female": p_f,
            "z_diff": z_diff,
            "p_diff": p_diff,
            "fdr_diff": fdr_diff,
        }

    if return_type == "structured":
        dtype = np.dtype([
            ("protein", "U256"),
            ("n_all", "i4"),
            ("n_male", "i4"),
            ("n_female", "i4"),
            ("rho_all", "f8"),
            ("p_all", "f8"),
            ("rho_male", "f8"),
            ("p_male", "f8"),
            ("rho_female", "f8"),
            ("p_female", "f8"),
            ("z_diff", "f8"),
            ("p_diff", "f8"),
            ("fdr_diff", "f8"),
        ])
        out = np.zeros(n_proteins, dtype=dtype)
        out["protein"] = protein_str
        out["n_all"] = n_all
        out["n_male"] = n_male
        out["n_female"] = n_female
        out["rho_all"] = rho_all
        out["p_all"] = p_all
        out["rho_male"] = rho_m
        out["p_male"] = p_m
        out["rho_female"] = rho_f
        out["p_female"] = p_f
        out["z_diff"] = z_diff
        out["p_diff"] = p_diff
        out["fdr_diff"] = fdr_diff
        return out

    if return_type == "dataframe":
        import pandas as pd  # standard in most scientific stacks
        return pd.DataFrame({
            "protein": protein_str,
            "n_all": n_all,
            "n_male": n_male,
            "n_female": n_female,
            "rho_all": rho_all,
            "p_all": p_all,
            "rho_male": rho_m,
            "p_male": p_m,
            "rho_female": rho_f,
            "p_female": p_f,
            "z_diff": z_diff,
            "p_diff": p_diff,
            "fdr_diff": fdr_diff,
        })

    raise ValueError("return_type must be one of: 'dict', 'structured', 'dataframe'")

In [ ]:
stats = spearman_fisherz_sex_diff(
    X,
    protein_names=protein_names,
    sex=sex,
    timepoint=month,
    male_label=0,
    female_label=1,
    min_n_per_sex=6,
    nan_policy="pairwise",
    return_type="dataframe",  # or "dict"/"structured"
)

# Example: top sex-difference hits
top = stats.sort_values(["fdr_diff", "p_diff"]).head(50)
top

In [ ]:
from __future__ import annotations

from typing import Optional, Tuple, Union

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.stats import rankdata


def visualize_sex_diff(
    results: pd.DataFrame,
    X: np.ndarray,
    protein_names: np.ndarray,
    sex: np.ndarray,
    timepoint: np.ndarray,
    *,
    fdr_cutoff: float = 0.05,
    male_label: Union[str, int] = "M",
    female_label: Union[str, int] = "F",
    male_color: str = "purple",
    female_color: str = "green",
    log2: bool = False,
    plots_per_row: int = 3,
    point_alpha: float = 0.65,
    point_size: float = 25.0,
    line_alpha: float = 0.9,
    figsize_per_panel: Tuple[float, float] = (3.8, 3.2),
    suptitle: Optional[str] = None,
    save_path: Optional[str] = None,
    dpi: int = 150,
) -> plt.Figure:
    """
    Visualize per-protein sex-difference results from ``spearman_fisherz_sex_diff``.

    For each protein whose FDR (fdr_diff) is <= ``fdr_cutoff``, one panel is drawn
    showing:
      - Male / female scatter points (timepoint on x, protein abundance on y)
      - A LOWESS-free linear trend line per sex (OLS on the raw / log2-transformed
        values, purely for visual guidance — not the Spearman rho line)
      - Title block with protein name, sex-specific Spearman rhos, and FDR

    Parameters
    ----------
    results : pd.DataFrame
        Output of ``spearman_fisherz_sex_diff(..., return_type="dataframe")``.
        Must contain columns: protein, rho_male, rho_female, fdr_diff.
    X : (n_samples, n_proteins) array
        Raw abundance matrix — same object passed to the stats function.
    protein_names : (n_proteins,) array-like
        Protein labels — same object passed to the stats function.
    sex : (n_samples,) array-like
        Sex labels — same object passed to the stats function.
    timepoint : (n_samples,) array-like
        Timepoints — same object passed to the stats function.
    fdr_cutoff : float
        Only proteins with fdr_diff <= this value are plotted.
    male_label / female_label : str or int
        Labels used to identify sexes in ``sex``.
    male_color / female_color : str
        Matplotlib colour strings for male / female points and trend lines.
    log2 : bool
        If True, apply log2(x + 1) transformation to the abundance values before
        plotting. The Spearman rho annotations always reflect the values computed
        by the stats function (not re-computed here).
    plots_per_row : int
        Number of panels per row in the figure grid.
    point_alpha : float
        Opacity for scatter points (0–1).
    point_size : float
        Size for scatter points (matplotlib ``s`` parameter).
    line_alpha : float
        Opacity for trend lines (0–1).
    figsize_per_panel : (width, height)
        Size in inches for each individual panel.
    suptitle : str, optional
        Overall figure title. Defaults to an auto-generated string.
    save_path : str, optional
        If provided, the figure is saved to this path (format inferred from
        extension, e.g. "output.png", "output.pdf").
    dpi : int
        Resolution used when saving.

    Returns
    -------
    matplotlib.figure.Figure
        The figure object (also displayed if in an interactive / notebook context).

    Examples
    --------
    >>> fig = visualize_sex_diff(
    ...     results=df_results,
    ...     X=X,
    ...     protein_names=protein_names,
    ...     sex=sex,
    ...     timepoint=timepoint,
    ...     fdr_cutoff=0.10,
    ...     log2=True,
    ...     plots_per_row=4,
    ... )
    """
    # ------------------------------------------------------------------ #
    # 0.  Coerce inputs
    # ------------------------------------------------------------------ #
    X = np.asarray(X, dtype=float)
    protein_names = np.asarray(protein_names)
    sex = np.asarray(sex)
    timepoint = np.asarray(timepoint, dtype=float)

    # Mirror the integer-label normalisation from the stats function
    if isinstance(male_label, (int, np.integer)) or isinstance(female_label, (int, np.integer)):
        sex = sex.astype(int)
        male_label = int(male_label)
        female_label = int(female_label)

    # ------------------------------------------------------------------ #
    # 1.  Filter results to significant proteins, preserving FDR order
    # ------------------------------------------------------------------ #
    sig = results[results["fdr_diff"] <= fdr_cutoff].copy()
    sig = sig.sort_values("fdr_diff")

    if sig.empty:
        raise ValueError(
            f"No proteins pass fdr_cutoff={fdr_cutoff}. "
            "Try a larger cutoff or check that 'fdr_diff' column is present."
        )

    # Map protein name → column index in X
    pname_to_idx = {str(p): i for i, p in enumerate(protein_names)}

    missing = [p for p in sig["protein"].astype(str) if p not in pname_to_idx]
    if missing:
        raise ValueError(
            f"{len(missing)} protein(s) in results not found in protein_names: "
            f"{missing[:5]}{'...' if len(missing) > 5 else ''}"
        )

    # ------------------------------------------------------------------ #
    # 2.  Build figure grid
    # ------------------------------------------------------------------ #
    n_panels = len(sig)
    n_cols = max(1, plots_per_row)
    n_rows = int(np.ceil(n_panels / n_cols))

    fig_w = figsize_per_panel[0] * n_cols
    fig_h = figsize_per_panel[1] * n_rows

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(fig_w, fig_h),
        squeeze=False,
    )

    # ------------------------------------------------------------------ #
    # 3.  Sex masks
    # ------------------------------------------------------------------ #
    male_mask = (sex == male_label)
    female_mask = (sex == female_label)

    t_male = timepoint[male_mask]
    t_female = timepoint[female_mask]

    # ------------------------------------------------------------------ #
    # 4.  Helper: OLS trend line
    # ------------------------------------------------------------------ #
    def _ols_line(
        tx: np.ndarray,
        yx: np.ndarray,
    ) -> Tuple[np.ndarray, np.ndarray]:
        """
        Return (x_line, y_line) for an OLS fit through finite (tx, yx) pairs.
        Spans the observed x range with 200 points.
        """
        m = np.isfinite(tx) & np.isfinite(yx)
        if m.sum() < 2:
            return np.array([]), np.array([])
        tx_f = tx[m]
        yx_f = yx[m]
        # Fit degree-1 polynomial (OLS)
        coeffs = np.polyfit(tx_f, yx_f, 1)
        x_line = np.linspace(tx_f.min(), tx_f.max(), 200)
        y_line = np.polyval(coeffs, x_line)
        return x_line, y_line

    # ------------------------------------------------------------------ #
    # 5.  Draw panels
    # ------------------------------------------------------------------ #
    def _fmt_rho(val) -> str:
        try:
            return f"{float(val):.3f}"
        except (TypeError, ValueError):
            return "nan"

    def _fmt_fdr(val) -> str:
        try:
            v = float(val)
            if v < 0.001:
                return f"{v:.2e}"
            return f"{v:.4f}"
        except (TypeError, ValueError):
            return "nan"

    axes_flat = axes.flatten()

    for ax_idx, (_, row) in enumerate(sig.iterrows()):
        ax = axes_flat[ax_idx]
        pname = str(row["protein"])
        col_idx = pname_to_idx[pname]

        y_all = X[:, col_idx].copy()
        if log2:
            # log2(x + 1) — safe for zero / near-zero values
            y_all = np.log2(np.maximum(y_all, 0) + 1)

        y_male = y_all[male_mask]
        y_female = y_all[female_mask]

        # ---- scatter ------------------------------------------------- #
        ax.scatter(
            t_male, y_male,
            color=male_color, alpha=point_alpha, s=point_size,
            label=f"Male (ρ={_fmt_rho(row['rho_male'])})",
            zorder=3, linewidths=0,
        )
        ax.scatter(
            t_female, y_female,
            color=female_color, alpha=point_alpha, s=point_size,
            label=f"Female (ρ={_fmt_rho(row['rho_female'])})",
            zorder=3, linewidths=0,
        )

        # ---- trend lines --------------------------------------------- #
        xl_m, yl_m = _ols_line(t_male, y_male)
        if xl_m.size:
            ax.plot(xl_m, yl_m, color=male_color, alpha=line_alpha,
                    linewidth=1.8, zorder=4)

        xl_f, yl_f = _ols_line(t_female, y_female)
        if xl_f.size:
            ax.plot(xl_f, yl_f, color=female_color, alpha=line_alpha,
                    linewidth=1.8, zorder=4)

        # ---- annotations --------------------------------------------- #
        fdr_str = _fmt_fdr(row["fdr_diff"])
        ax.set_title(
            f"{pname}\n"
            f"ρ♂={_fmt_rho(row['rho_male'])}  ρ♀={_fmt_rho(row['rho_female'])}\n"
            f"FDR={fdr_str}",
            fontsize=8, loc="left", pad=4,
        )

        ax.set_xlabel("Timepoint", fontsize=7)
        y_label = "log₂(abundance + 1)" if log2 else "Abundance"
        ax.set_ylabel(y_label, fontsize=7)
        ax.tick_params(labelsize=6)
        # ax.legend(fontsize=6, framealpha=0.6, loc="best")
        ax.spines[["top", "right"]].set_visible(False)

    # ------------------------------------------------------------------ #
    # 6.  Hide unused axes
    # ------------------------------------------------------------------ #
    for ax_idx in range(n_panels, len(axes_flat)):
        axes_flat[ax_idx].set_visible(False)

    # ------------------------------------------------------------------ #
    # 7.  Figure-level title and layout
    # ------------------------------------------------------------------ #
    if suptitle is None:
        suptitle = (
            f"Sex-differential protein–timepoint correlations  "
            f"(FDR ≤ {fdr_cutoff}, n={n_panels})"
        )
    fig.suptitle(suptitle, fontsize=10, y=1.01)
    fig.tight_layout()

    # ------------------------------------------------------------------ #
    # 8.  Optionally save
    # ------------------------------------------------------------------ #
    if save_path is not None:
        fig.savefig(save_path, dpi=dpi, bbox_inches="tight")

    return fig

In [ ]:
fig = visualize_sex_diff(
    results=stats,          # dataframe from the stats function
    X=X,                 # same abundance matrix
    protein_names=protein_names,   # same protein names array
    sex=sex,             # same sex array
    timepoint=month, # same timepoint array
    fdr_cutoff=0.05,
    log2=True,
    plots_per_row=6,
    male_label=0,
    female_label=1,
    male_color='orangered',
    female_color='purple'
)